# Implementación de controladores digitales con Julia



En este script vamos a realizar la implementación de un controlador digital de un controlador digital para el ángulo del motor DC del laboratorio. Consideremos que en un experimento de identificación obtuvimos el siguiente modelo:

$$G(s)=\dfrac{1232.9}{s(0.33s+1)}$$

Asimismo, para este sistema hemos obtenido el siguiente controlador, usando un método de diseño frecuencial:

$$C(s)=\dfrac{0.05506(s + 26)(s + 3.055)}{(s^2 + 40.72*s + 613.4)}$$

A continuación definimos estos valores en Julia.

*Para correr este notebook debe instalar los paquetes Roots, RobustAndOptimalControl y Printf, además de los que usa normalmente en el curso de control*


In [ ]:
# Incluimos el paquete de control de este notebook que extiende el modulo de la clase
include("ControlUNPlus.jl") 
s= tf("s")
s = tf("s");
G = 1232.9/(s*(0.33*s+1));
Controller = 0.05506*(s + 26)*(s + 3.055)/(s^2 + 40.72*s + 613.4)
#Controller = 0.039597*(s+43.11)*(s+4.222)*(s+3.333)*(s^2 + 21.02*s + 256)/(s* (s+19.05)* (s+11.2) *(s^2 + 22.81*s + 544.2))

## Paso 1: selección del tiempo de muestreo


Para encontrar el tiempo de muestreo apropiado, calculamos la respuesta en frecuencia del sistema de lazo cerrado.

In [ ]:
T = feedback(Controller*G,1)
fb = bandwidth(T)/(2*pi)
fh = 10*fb
hmax = 1/fh;
println("El ancho de banda del sistema es $(round(fb, digits=2)) Hz, la frecuencia de muestreo debe ser al menos  $(round(fh, digits=2)) Hz")
println("El tiempo de muestreo tiene que ser menor a $(round(hmax, digits=2)) segundos")
println("El tiempo de muestreo de 0.01 segundos es una buena elección")
bodemag(T)

## Paso 2: Discretización del controlador

Para discretizar el controlador se usa la transformación bilineal definida como:

$$C_d(z)= \dfrac{2}{h} \dfrac{z-1}{z+1}$$

 Esto lo hacemos con los siguientes comandos:

In [ ]:
h = 0.01;
Cdz = c2d(Controller, h, :tustin)

## Paso 3: conversión a variables de estado
En este paso encontramos una realización en variables de estado para el controlador discreto de la forma:

$$x[n+1]= Ax[n] + B e[n]$$
$$u[n] = Cx[n] + D x[n]$$

Esto lo hacemos con el siguientes comando que permiten obtener una representación modal, la cual tiene buenas características numéricas.


In [ ]:
Cdz_estado, S, E = modal_form(ss(Cdz))

## Paso 4: Control Anti-windup
Calculamos la ganancia L del control de antiwindup con el método de Kalman, mediante el comando siguiente

In [ ]:
L = kalman(Cdz_estado, 100, 1)

## Paso 5: Implementación del código del controlador
Obtenemos las matrices A,B,C y D del controlador discretizado, representado en variables de estado:

In [ ]:
A, B, C, D = ssdata(Cdz_estado);

Obtenemos las matrices $A_{aw}$ y $B_{aw}$

In [ ]:
Baw = B - L*D;
Aaw = A - L*C;
println(L)
eigvals(Aaw)

Por último, utilizamos la siguiente función, creada para esta guia, que escribe automaticamente el código del controlador en el archivo `controller.h`.

In [ ]:
generateCode(Aaw, Baw, C, D, L) 